In [1]:
import os
os.environ['USER_AGENT'] = 'BA_GPT/1.0 (contact: walrus_z@hotmail.com)'

In [2]:
import os
import sys
import yaml
import types
import pprint
import logging
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat import EVENT_LOGGER_NAME, TRACE_LOGGER_NAME
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.memory.canvas import TextCanvasMemory
from autogen_ext.models.openai import OpenAIChatCompletionClient
from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.instrumentation.openai import OpenAIInstrumentor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor


# Project import paths
# Get the current directory (which is project/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

from agents_creator import create_agents_from_config
from rag_helpers import rag_indexer_runner
from model_loader import load_model_client


In [3]:
logging.basicConfig(level=logging.WARNING)

# For trace logging.
trace_logger = logging.getLogger(TRACE_LOGGER_NAME)
trace_logger.addHandler(logging.StreamHandler())
trace_logger.setLevel(logging.DEBUG)

# For structured message logging, such as low-level messages between agents.
event_logger = logging.getLogger(EVENT_LOGGER_NAME)
event_logger.addHandler(logging.StreamHandler())
event_logger.setLevel(logging.DEBUG)

In [ ]:
# Set up telemetry span exporter.
otel_exporter = OTLPSpanExporter(endpoint="http://localhost:4317", insecure=True)
span_processor = BatchSpanProcessor(otel_exporter)

# Set up telemetry trace provider.
tracer_provider = TracerProvider(resource=Resource({"service.name": "autogen-test-agentchat"}))
tracer_provider.add_span_processor(span_processor)
trace.set_tracer_provider(tracer_provider)

# Instrument the OpenAI Python library
OpenAIInstrumentor().instrument()

# we will get reference this tracer later using its service name
# tracer = trace.get_tracer("autogen-test-agentchat")

ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
ERROR:opentelemetry.exporter.otlp.proto.grpc.exporter:Failed to export trace

In [5]:
rag_vector_memory = await rag_indexer_runner('/run/media/jun/Jun 6T/Code/TFM_BA_GPT/lib/rag_config.yaml')

✅ All documents and URLs are already indexed.


In [6]:
# 1. Create the canvas memory and tool:
text_canvas_memory = TextCanvasMemory()
update_file_tool = text_canvas_memory.get_update_file_tool()

# 2. Register the tool as 'canvas.update_file_tool' for YAML import:
canvas = types.SimpleNamespace()
canvas.update_file_tool = update_file_tool
sys.modules['canvas'] = canvas  # This enables the tool import by your agent factory!

In [ ]:
model_client = load_model_client("azure")

In [8]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'model': 'google/gemma-3-27b', 'api_key': 'LMStudio_key', 'model_info': {'vision': False, 'function_calling': True, 'json_output': True, 'family': 'gemma', 'structured_output': True, 'multiple_system_messages': True}, 'base_url': 'http://127.0.0.1:1234/v1'}
create_args: {'model': 'google/gemma-3-27b'}


In [9]:
# Construct the absolute path to project/lib/agents_config.yaml user_proxy agent
AGENTS_CONFIG = os.path.abspath(os.path.join(notebook_dir, '..', 'lib', 'agents_config_user_proxy_agent.yaml'))
with open(AGENTS_CONFIG, "r") as f:
        config_yaml = yaml.safe_load(f)
agent_names = list(config_yaml.keys())

user, agents = await create_agents_from_config(
    AGENTS_CONFIG,
    agent_names=agent_names,
    model_client=model_client,
    vector_memory=rag_vector_memory,
    canvas_memory=text_canvas_memory,
    input_func=input
)

In [10]:
# 1️⃣ Grab the existing writer agent
old_writer = agents["functional_spec_writer_agent"]

# 2️⃣ Wrap the sub-agents as tools
use_case_tool = AgentTool(agent=agents["use_case_agent"])
glossary_tool = AgentTool(agent=agents["glossary_agent"])

# 3️⃣ Collect the writer's current tools (if any)
existing_tools = getattr(old_writer, "_tools", []) or []

# 4️⃣ Combine tools: existing ones + new agent-tools
combined_tools = existing_tools + [use_case_tool, glossary_tool]

# 5️⃣ Extract the system message text (AutoGen stores it as a list of SystemMessage objects)
system_message_text = ""
if hasattr(old_writer, "_system_messages") and old_writer._system_messages:
    system_message_text = old_writer._system_messages[0].content  # first SystemMessage content

# 6️⃣ Rebuild functional_spec_writer_agent with combined tools
agents["functional_spec_writer_agent"] = AssistantAgent(
    name=old_writer._name,
    description=old_writer._description,
    model_client=old_writer._model_client,
    memory=old_writer._memory,
    tools=combined_tools,
    reflect_on_tool_use=old_writer._reflect_on_tool_use,
    system_message=system_message_text,
    model_client_stream=old_writer._model_client_stream,
)

print("✅ functional_spec_writer_agent rebuilt with use_case_tool and glossary_tool attached.")


✅ functional_spec_writer_agent rebuilt with use_case_tool and glossary_tool attached.


In [11]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    pprint.pprint(vars(agent))




--- Agent: planning_agent ---
{'_description': 'The orchestrator agent responsible for breaking down a new '
                 'task into subtasks, assigning them to the right agents, and '
                 'ensuring the workflow progresses smoothly until completion.',
 '_handoff_tools': [],
 '_handoffs': {},
 '_is_running': False,
 '_max_tool_iterations': 1,
 '_memory': None,
 '_metadata': {},
 '_model_client': <autogen_ext.models.openai._openai_client.OpenAIChatCompletionClient object at 0x7f525d172120>,
 '_model_client_stream': True,
 '_model_context': <autogen_core.model_context._unbounded_chat_completion_context.UnboundedChatCompletionContext object at 0x7f525d172cf0>,
 '_name': 'planning_agent',
 '_output_content_type': None,
 '_output_content_type_format': None,
 '_reflect_on_tool_use': False,
 '_structured_message_factory': None,
 '_system_messages': [SystemMessage(content='You are the PLANNING AGENT, the coordinator for this multi-agent system.\n\n**YOUR MISSION**\n  Break dow

In [12]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    print("Tools:")
    for tool in agent._tools:
        print(f"Tool Name: {tool.name}")
        print(f"Description: {tool.description}\n")



--- Agent: planning_agent ---
Tools:

--- Agent: interviewer_agent ---
Tools:
Tool Name: save_conversation_to_jsonl
Description: 
Saves either:
1) A Q&A pair (interviewer + user) in one atomic JSONL write, OR
2) A generic list of records.



--- Agent: summarizer_agent ---
Tools:

--- Agent: functional_spec_writer_agent ---
Tools:
Tool Name: write_md_file_tool
Description: 
Writes the functional specification to a Markdown file.


Tool Name: use_case_agent
Description: An agent dedicated to generating Use Cases and User Stories from the project information.

Tool Name: glossary_agent
Description: An agent dedicated to building a Glossary of terms from the gathered requirements and project information.


--- Agent: functional_spec_checker_agent ---
Tools:
Tool Name: read_md_file_tool
Description: 
Reads the entire content of a Markdown file and returns it as a string.

Args:
    file_path (str): Optional path to the Markdown spec (defaults to DEFAULT_MD_SPEC_PATH).

Returns:
    str: T

In [13]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.

HARD ROUTING RULES
    If the last message contains a line starting with `QUESTION:` from interviewer_agent, select: user.
    If the last message is from user and the previous message from interviewer_agent contained `QUESTION:`, select: interviewer_agent.
    If planning_agent has NOT assigned the next task in its last message, select: planning_agent.
    If interviewer_agent returned `LOG_RETRY`, select: interviewer_agent (to retry logging).
    Otherwise, follow the planner's most recent explicit single assignment. Only select one agent.

SAFETY
    Do NOT select any agent other than 'user' while the planner's state indicates awaiting_user_reply = true (if present in context).
"""

In [14]:
# Order agents (workflow order), user_proxy last
agent_order = [
    "planning_agent",
    "interviewer_agent",
    "summarizer_agent",
    "functional_spec_writer_agent",
    "functional_spec_checker_agent",
    "diagram_creator_agent",
    "mermaid_code_reviewer_agent",
]
member_agents = [agents[name] for name in agent_order if name in agents] + [user]

In [15]:
termination = MaxMessageTermination(100000000) | TextMentionTermination("TERMINATE")

# --- Use SelectorGroupChat ---
team = SelectorGroupChat(
    member_agents,
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False
)

In [16]:
task = "Interview a customer to collect requirements and scope of a new product then write the functional specifications"

In [17]:
runtime = SingleThreadedAgentRuntime(
            tracer_provider=trace.NoOpTracerProvider(),  # Disable telemetry for runtime.
        )
runtime.start()

In [ ]:
await Console(team.run_stream(task=task))

await runtime.stop()

await model_client.close()

Model selected a valid name: planning_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: planning_agent (attempt 1)
Selected speaker: planning_agent
DEBUG:autogen_agentchat:Selected speaker: planning_agent


---------- TextMessage (user) ----------
Interview a customer to collect requirements and scope of a new product then write the functional specifications
---------- ModelClientStreamingChunkEvent (planning_agent) ----------
Okay, let's begin!

Here's the initial task breakdown:

1. interviewer_agent : Start gathering requirements from the user regarding the new product.

**Task List:**
interviewer_agent: pending
user: not started
summarizer_agent: not started
functional_spec_writer_agent: not started
functional_spec_checker_agent: not started
diagram_creator_agent: not started
mermaid_code_reviewer_agent: not started

awaiting_user_reply = true



Model selected a valid name: interviewer_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: interviewer_agent (attempt 1)
Selected speaker: interviewer_agent
DEBUG:autogen_agentchat:Selected speaker: interviewer_agent


---------- MemoryQueryEvent (interviewer_agent) ----------
[MemoryContent(content='Requirements Gathering - Most Asked Questions | Requiment     +321 123 4567[email\xa0protected]My AccountWhy Requiment?FeaturesGuided ProcessOutput ReportsCreate WireframesUpdate EasilyTask GenerationPricingWho We AreResourcesBlogVideo WalkthroughBitesize DemosFAQBook a DemoLog InStart Free TrialWhy Requiment?FeaturesGuided ProcessOutput ReportsCreate WireframesUpdate EasilyTask GenerationPricingWho We AreResourcesBlogVideo WalkthroughBitesize DemosFAQBook a DemoStart Free TrialLog inRequirements', mime_type='MemoryMimeType.TEXT', metadata={'mime_type': 'MemoryMimeType.TEXT', 'chunk_index': 0, 'source': 'https://www.requiment.com/the-most-asked-questions-about-requirements-gathering/', 'score': 0.5049756169319153, 'id': '943e53d4-63ac-47aa-a8d9-933dbbb47476'}), MemoryContent(content='55 Business Analyst Requirements Gathering Interview Questions and Answers | Requiment     +321 123 4567[email\xa0protecte

Model selected a valid name: user (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: user (attempt 1)
Selected speaker: user
DEBUG:autogen_agentchat:Selected speaker: user


---------- TextMessage (user) ----------
The new product is an AI dialer that calls customers to offer travel packages. If the customer accepts, the call transfers to a human. If declined, the AI attempts to convince the customer up to three additional times before ending the call


... (153 outputs left out)

---------- TextMessage (planning_agent) ----------



/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_core/_routed_agent.py:268: UserWarning: No text content or tool calls are available. Model returned empty result.
  return_value = await func(self, message, ctx)  # type: ignore
Model failed to select a valid name: documentation_generator_agent (attempt 1)
DEBUG:autogen_agentchat:Model failed to select a valid name: documentation_generator_agent (attempt 1)
Model failed to select a valid name: documentation_generator_agent (attempt 2)
DEBUG:autogen_agentchat:Model failed to select a valid name: documentation_generator_agent (attempt 2)
Model failed to select a valid name: documentation_generator_agent (attempt 3)
DEBUG:autogen_agentchat:Model failed to select a valid name: documentation_generator_agent (attempt 3)
Model failed to select a speaker after 3, using the previous speaker.
Selected speaker: planning_agent
DEBUG:autogen_agentchat:Selected speaker: planning_agent


---------- TextMessage (planning_agent) ----------



/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_core/_routed_agent.py:268: UserWarning: No text content or tool calls are available. Model returned empty result.
  return_value = await func(self, message, ctx)  # type: ignore
Model failed to select a valid name: documentation_generator_agent (attempt 1)
DEBUG:autogen_agentchat:Model failed to select a valid name: documentation_generator_agent (attempt 1)
Model failed to select a valid name: documentation_generator_agent (attempt 2)
DEBUG:autogen_agentchat:Model failed to select a valid name: documentation_generator_agent (attempt 2)
Model failed to select a valid name: documentation_generator_agent (attempt 3)
DEBUG:autogen_agentchat:Model failed to select a valid name: documentation_generator_agent (attempt 3)
Model failed to select a speaker after 3, using the previous speaker.
Selected speaker: planning_agent
DEBUG:autogen_agentchat:Selected speaker: planning_agent
/run/media/jun/Jun 6T/Code/TFM_BA_GP